# 02 — Forced alignment лирики на htdemucs vocal

Главный технический угол: знаем лирику → wav2vec2-CTC + `torchaudio.functional.forced_align` → word timestamps.

WER = 0 by construction. Полезные метрики:
- **coverage** — какая доля слов лирики реально нашлась в аудио (низкий = кавер/другая версия).
- **mean confidence** — средняя prob фреймов на пути выравнивания (низкая = артефакты demucs / back-vocals).
- **timing offset** vs baseline — насколько сместились границы слов относительно faster-whisper.

Запускать на **Colab T4**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU не подключен → Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/RezPoint/unbake-test.git
%cd unbake-test
!pip install -q transformers torchaudio librosa jiwer pydantic requests

In [ ]:
!python notebooks/download_dataset.py
!ls data/raw/ru/

## Лирика

В репо лирик нет (`data/` в .gitignore — Genius-копирайт). Вставляем строкой.

In [ ]:
LYRICS = """Самый редкий вид, но самый худший браконьер
Спорим, она вместит себе в глотку револьвер?
Спорим, что ты больше не покинешь этот сквер?
Я играю с ее киской, детка, где твой кавалер?
Я врубаю Дафт Панк, она крутит мне блант
Покидаем клуб, в руке Совиньон-блан
Отлип от ее губ, залипаю в экран
Я кидаю деньги в воздух — ты поймаешь их сам
Эй, детка, ты модель или подделка?
Сейчас не важно, важно только, что не целка
А если кто-то из твоих захочет знать
Я найду еще причины, чтобы положить их спать
Она качает гривой, нельзя быть такой игривой
Выдыхаю в потолок испарения сативы
Кажусь хуже, чем я есть, заставляю тебя есть
Я держу при себе суку, что не может с меня слезть
Это все дико, например
Мы курим дико, например
Мы виснем дико, например
Ты не сумел, но я дико отымел
Это все дико, например
Мы курим дико, например
Мы виснем дико, например
Ты не сумел, но я дико отымел
Извини за прямоту
Но даже в этом клубе все когда-нибудь умрут
Мои парни мне сказали, что мне нужно охладить
И я держу это на низком, ведь та сука троглодит
Повсюду лужи крови, помогу ей обойти
Но ты беги-беги-беги
Марриотт с меня хуеет, трижды место преступления
Моя тишка говорит мне: Я ошибка поколения
Это все дико, например
Мы курим дико, например
Мы виснем дико, например
Ты не сумел, но я дико отымел
Это все дико, например
Мы курим дико, например
Мы виснем дико, например
Ты не сумел, но я дико отымел"""

In [ ]:
from src.align import align

TRACK = 'data/raw/ru/Pharaoh - Дико, например.m4a'
transcript, tele = align(TRACK, LYRICS, language='ru')
print(tele)

In [ ]:
# Первые 30 выровненных слов
for w in transcript.words[:30]:
    print(f'{w.start:6.2f} → {w.end:6.2f}  {w.text:20s} (conf={w.confidence:.2f})')

In [ ]:
# Слова с низкой confidence — кандидаты на кавер/артефакт
low = [w for w in transcript.words if (w.confidence or 0) < 0.5]
print(f'low-conf: {len(low)} / {len(transcript.words)}')
for w in low[:20]:
    print(f'  {w.start:6.2f}  {w.text}  conf={w.confidence:.2f}')

In [ ]:
import json, dataclasses
with open('/content/align_pharaoh_ru.json', 'w', encoding='utf-8') as f:
    f.write(transcript.model_dump_json(indent=2))
with open('/content/align_pharaoh_ru.telemetry.json', 'w', encoding='utf-8') as f:
    json.dump(dataclasses.asdict(tele), f, indent=2, ensure_ascii=False)
print('saved → /content/align_pharaoh_ru.json')
print('saved → /content/align_pharaoh_ru.telemetry.json')

## Что смотреть

- `coverage` — на знакомой студийной версии должно быть ≥ 0.95. < 0.7 = кавер / другая лирика / битый vocal.
- `mean_confidence` — у XLSR-53-russian обычно 0.6-0.85 на чистом вокале. < 0.4 = htdemucs съел согласные / back-vocals задавили.
- `rtf` — XLSR-53 на T4 даёт ≈ 0.04-0.06 (без beam, single-pass). В сумме с baseline всё ещё ниже потолка cost.